In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
from train import train_classifier, test_classifier
from model import LatentClassifier, Classifier
device = "cuda:0"

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(
        lambda x: x.view(-1)
    )
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
def run_control_experiments(train_loader, test_loader, device):
    training_times = 5
    results = []    
    train_accuracies = []
    test_accuracies = []
    for t in range(training_times):
        print(f"run={t+1}")
        model = Classifier()
        model = model.to(device)
        train_classifier(train_loader, model, 100, f"classifier{t}.pth")
        train_acc = test_classifier(train_loader, model)
        test_acc = test_classifier(test_loader, model)
        train_accuracies.append(train_acc)
        test_accuracies.append(test_acc)
    result = {
        "label": f"non_latent_classifier",
        "mean_accuracy_train":
            np.mean(train_accuracies),
        "std_accuracy_train":
            np.std(train_accuracies),
        "mean_accuracy_test":
            np.mean(test_accuracies),
        "std_accuracy_test":
            np.std(test_accuracies),
    }
    results.append(result)
    return results

In [ ]:
def run_z_dim_experiments(train_loader, test_loader, device):
    z_dim_experiments = {"z_dim": [256, 512, 1024, 2048]}
    training_times = 5
    results = []
    for z_dim in z_dim_experiments["z_dim"]:
        train_accuracies = []
        test_accuracies = []
        for t in range(training_times):
            print(f"z_dim={z_dim} run={t+1}")
            model = LatentClassifier(z_dim=z_dim, is_plus_z=True, is_W_orthogonal=True)
            model = model.to(device)
            train_classifier(train_loader, model, 100, f"1latent_classifier{t}.pth")
            train_acc = test_classifier(train_loader, model)
            test_acc = test_classifier(test_loader, model)
            train_accuracies.append(train_acc)
            test_accuracies.append(test_acc)
        result = {
            "label": f"z={z_dim}",
            "mean_accuracy_train":
                np.mean(train_accuracies),
            "std_accuracy_train":
                np.std(train_accuracies),
            "mean_accuracy_test":
                np.mean(test_accuracies),
            "std_accuracy_test":
                np.std(test_accuracies),
        }
        results.append(result)
    return results

In [ ]:
def run_operation_experiments(train_loader, test_loader, device, z_dim=512):
    training_times = 5
    operation_experiments = {"is_plus_z": [True, False], "is_W_orthogonal": [True, False]}
    results = []
    for is_plus_z in operation_experiments["is_plus_z"]:
        for is_W_orthogonal in operation_experiments["is_W_orthogonal"]:
            train_accuracies = []
            test_accuracies = []
            for t in range(training_times):
                print(
                    f"is_plus_z={is_plus_z} "
                    f"is_W_orthogonal={is_W_orthogonal} "
                    f"run={t+1}"
                )
                model = LatentClassifier(
                    z_dim=z_dim,
                    is_plus_z=is_plus_z,
                    is_W_orthogonal=is_W_orthogonal,
                )
                model = model.to(device)
                train_classifier(
                    train_loader,
                    model,
                    100,
                    f"2latent_classifier{t}.pth",
                )
                train_acc = test_classifier(
                    train_loader,
                    model,
                )
                test_acc = test_classifier(
                    test_loader,
                    model,
                )
                train_accuracies.append(train_acc)
                test_accuracies.append(test_acc)
            result = {
                "label": f"plus_z={is_plus_z}\n W_orthogonal={is_W_orthogonal}",
                "mean_accuracy_train": np.mean(train_accuracies),
                "std_accuracy_train": np.std(train_accuracies),
                "mean_accuracy_test": np.mean(test_accuracies),
                "std_accuracy_test": np.std(test_accuracies),
            }
            results.append(result)
    return results

In [ ]:
def plot_results(results, title, save_path):
    labels = [r["label"] for r in results]
    mean_train = [r["mean_accuracy_train"]for r in results]
    std_train = [r["std_accuracy_train"]for r in results]
    mean_test = [r["mean_accuracy_test"]for r in results]
    std_test = [r["std_accuracy_test"]for r in results]
    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    train_bars = ax.bar(
        x - width/2,
        mean_train,
        width,
        yerr=std_train,
        capsize=5,
        label="Train Accuracy",
    )
    
    test_bars = ax.bar(
        x + width/2,
        mean_test,
        width,
        yerr=std_test,
        capsize=5,
        label="Test Accuracy",
    )
    
    for bar, mean, std in zip(test_bars, mean_test, std_test):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            mean+std,
            f"{mean:.3f}±{std:.3f}",
            ha='center',
            va='bottom',
        )
    for bar, mean, std in zip(train_bars, mean_train, std_train):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            mean+std,
            f"{mean:.3f}±{std:.3f}",
            ha='center',
            va='bottom',
        )
    ax.set_ylim(min(mean_test+mean_train)-0.025, 1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("Accuracy")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
    plt.close()

In [ ]:
control_results = run_control_experiments(train_loader, test_loader, device)
z_dim_results = run_z_dim_experiments(train_loader, test_loader, device)
z_dim_results += control_results
operation_results = run_operation_experiments(train_loader, test_loader, device)
operation_results += control_results

In [ ]:
plot_results(z_dim_results, "z_dim_experiments", "./z_dim_results.jpg")

In [ ]:
plot_results(operation_results, "operation_experiments", "./operation_results.jpg")